In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jigsaw_agile_community_rules_path = kagglehub.competition_download('jigsaw-agile-community-rules')
level14taken_jigsaw_2m_reddit_unlabelled_path = kagglehub.dataset_download('level14taken/jigsaw-2m-reddit-unlabelled')
# hiranorm_jigsaw_packages2_path = kagglehub.notebook_output_download('hiranorm/jigsaw-packages2')
# qwen_lm_qwen2_5_transformers_14b_instruct_gptq_int4_1_path = kagglehub.model_download('qwen-lm/qwen2.5/Transformers/14b-instruct-gptq-int4/1')
yangjiahua_lora_14b_gptq_1epoch_r32_keras_default_1_path = kagglehub.model_download('yangjiahua/lora_14b_gptq_1epoch_r32/Keras/default/1')
noizersam_all_minilm_l12_v2_pytorch_l12_v2_1_path = kagglehub.model_download('noizersam/all-minilm-l12-v2/PyTorch/l12-v2/1')

print('Data source import complete.')


100%|██████████| 695k/695k [00:00<00:00, 785kB/s]

Extracting files...


Using Colab cache for faster access to the 'jigsaw-2m-reddit-unlabelled' dataset.
Data source import complete.


In [ ]:
!uv pip install --system  'trl==0.21.0' 'optimum==1.27.0' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system  'triton==3.2.0'
!uv pip install --system  'clean-text'
!uv pip install --system  -U --no-deps 'peft==0.17.1' 'accelerate==1.10.1' 'datasets==4.0.0'

Using Python 3.12.11 environment at: /usr
Resolved 153 packages in 1.48s
Prepared 50 packages in 31.50s
Uninstalled 12 packages in 690ms
Installed 50 packages in 221ms
 + astor==0.8.1
 + bitsandbytes==0.46.1
 + blake3==1.0.7
 + cbor2==5.7.0
 + compressed-tensors==0.10.2
 + deepspeed==0.17.4
 + depyf==0.19.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.13
 + fastapi-cloud-cli==0.3.0
 + gguf==0.17.1
 + hjson==3.1.0
 + httptools==0.6.4
 + interegular==0.3.3
 - lark==1.3.0
 + lark==1.2.2
 + llguidance==0.7.30
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.10.12
 + logits-processor-zoo==0.2.1
 + mistral-common==1.8.5
 + msgspec==0.19.0
 + ninja==1.13.0
 - numba==0.60.0
 + numba==0.61.2
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.5.1.17
 - nvidia-cusparselt-cu12==0.7.1
 + nvidia-cusparselt-cu12==0.6.3
 - nvidia-nccl-cu12==2.27.3
 + nvidia-nccl-cu12==2.26.2
 - openai==1.109.1
 + openai==1.90.0
 + optimum==1.27.0
 + outline

In [ ]:
import os
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
import torch
import vllm
import numpy as np
from vllm.lora.request import LoRARequest
import argparse
from scipy.special import softmax

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4'#f"{qwen_lm_qwen2_5_transformers_14b_instruct_gptq_int4_1_path}"
LORA_PATH = f"{yangjiahua_lora_14b_gptq_1epoch_r32_keras_default_1_path}"

os.environ["VLLM_USE_V1"] = "0"

# 2. Prepare Model

# 3. Prepare Prompt

In [ ]:
df = pd.read_csv(f"{jigsaw_agile_community_rules_path}/test.csv")
df_train= pd.read_csv(f'{jigsaw_agile_community_rules_path}/train.csv')
unlabelled_df= pd.read_csv(f'{level14taken_jigsaw_2m_reddit_unlabelled_path}/reddit-removal-log.csv')
print(df.shape)
df.head()

(10, 8)


,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2
0,2029,NEW RAP GROUP 17. CHECK US OUT https://soundcl...,"No Advertising: Spam, referral links, unsolici...",hiphopheads,"Hey, guys, just wanted to drop in and invite y...",Cum Swallowing Hottie Katrina Kaif Cartoon Xvi...,SD Stream Eng - [Chelsea TV USA](http://soccer...,HD Streams: |[ENG HD Stoke vs Manchester Unite...
1,2030,Make your life comfortable. Get up to 15% Disc...,No legal advice: Do not offer or request legal...,AskReddit,Get a lawyer and get the security camera foota...,That isn't drastic. You tried reaching out to ...,So what are you going to do with the insurance...,It's just for Austria & Germany. If you still ...
2,2031,Kickin' ass and selling underwear!\nJust made ...,"No Advertising: Spam, referral links, unsolici...",gonewild,Good story my friend. Check out my blog at ht...,If you know what exactly you need then you don...,CENTIPEDES\n\nSOME BASED PATRIOTS HAVE CREATED...,[So great! Thanks for sharing.](http://www.che...
3,2032,watch hooters best therein http://clickan...,"No Advertising: Spam, referral links, unsolici...",personalfinance,"Earn 50,000 bonus points with Chase Sapphire P...","Cool, front page! I made this print along with...",[Full HD Movie Online Free](http://www.flickma...,* Karambit Black Pearl\n* 0.02137822 Float (un...
4,2033,bitches for free at this point show all h...,"No Advertising: Spam, referral links, unsolici...",Showerthoughts,code free tyrande --->>> [Imgur](http://i.imgu...,My trade link\nhttps://steamcommunity.com/trad...,**HD** [ mio Stadium 102 HD](http://www.genti....,Infographics is an incredible method for showi...


In [ ]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [ ]:
augmented_train = add_data(df_train)
augmented_test = add_data(df)

augmented_texts = df_train['rule'].str.cat(df_train['body'], sep=' [SEP] ').tolist() + augmented_train[0] + augmented_test[0]
augmented_labels = df_train['rule_violation'].astype(float).tolist() + augmented_train[1] + augmented_test[1]

augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before dedup: {augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print(f'After dedup: {augmented_df.shape}')
augmented_df['rule'] = augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body'] = augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map = {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id'] = augmented_df.rule.str.lower().map(rule_map)

print(augmented_df.head())

Before dedup: (10185, 2)
After dedup: (1875, 2)
                                                text  label  \
0  No Advertising: Spam, referral links, unsolici...    1.0   
1  No Advertising: Spam, referral links, unsolici...    1.0   
2  No Advertising: Spam, referral links, unsolici...    0.0   
3  No Advertising: Spam, referral links, unsolici...    0.0   
4  No Advertising: Spam, referral links, unsolici...    1.0   

                                                rule  \
0  No Advertising: Spam, referral links, unsolici...   
1  No Advertising: Spam, referral links, unsolici...   
2  No Advertising: Spam, referral links, unsolici...   
3  No Advertising: Spam, referral links, unsolici...   
4  No Advertising: Spam, referral links, unsolici...   

                                                body  rule_id  
0  \n\nIf you have some free time on your hands, ...        0  
1  \n\nplease visit http://www.shifadental.net/te...        0  
2  \n\nSD | [ English Stream 1 Arsenal vs To

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search, dot_score
from tqdm import tqdm

def clean_text(text):
    if pd.isna(text) or text == "":
        return ""
    return text

def get_positives_from_augmented_df(augmented_df):
    positives_by_rule = {}
    for rule in augmented_df['rule'].unique():
        rule_data = augmented_df[augmented_df['rule'] == rule]
        positives = rule_data[rule_data['label'] >= 0.5]['body'].tolist()
        positives = [p for p in positives if pd.notna(p) and len(str(p).strip()) > 0]
        positives = list(set(positives))
        positives_by_rule[rule] = positives
        print(f"Rule '{rule[:50]}...': {len(positives)} positive examples")
    return positives_by_rule

def find_similar_unlabelled_examples(positive_examples, unlabelled_candidates,
                                     embedding_model, sample_size, similarity_threshold=0.3):
    if len(positive_examples) == 0 or len(unlabelled_candidates) == 0 or sample_size < 10:
        return pd.DataFrame()

    if len(unlabelled_candidates) > 50000:
        unlabelled_candidates = unlabelled_candidates.sample(n=50000, random_state=42)

    clean_positives = [clean_text(text) for text in positive_examples]
    candidate_bodies = unlabelled_candidates['body'].fillna('').astype(str).tolist()
    clean_candidates = [clean_text(text) for text in candidate_bodies]

    print(f"  Encoding {len(clean_positives)} positives and {len(clean_candidates)} candidates...")

    positive_embeddings = embedding_model.encode(
        sentences=clean_positives,
        batch_size=128*8,
        convert_to_tensor=True,
        device="cuda",
        normalize_embeddings=True,
        show_progress_bar=False
    )

    candidate_embeddings = embedding_model.encode(
        sentences=clean_candidates,
        batch_size=128*8,
        convert_to_tensor=True,
        device="cuda",
        normalize_embeddings=True,
        show_progress_bar=False
    )

    search_results = semantic_search(
        query_embeddings=positive_embeddings,
        corpus_embeddings=candidate_embeddings,
        top_k=min(sample_size * 3, len(clean_candidates)),
        score_function=dot_score
    )

    candidate_scores = {}
    for pos_idx, pos_results in enumerate(search_results):
        for result in pos_results:
            candidate_idx = result['corpus_id']
            similarity = result['score']
            if similarity >= similarity_threshold:
                if candidate_idx not in candidate_scores:
                    candidate_scores[candidate_idx] = []
                candidate_scores[candidate_idx].append(similarity)

    candidate_mean_scores = {
        idx: np.mean(scores) for idx, scores in candidate_scores.items()
    }

    top_candidates = sorted(candidate_mean_scores.items(),
                           key=lambda x: x[1], reverse=True)[:sample_size]

    if not top_candidates:
        print(f"  No similar candidates found (threshold={similarity_threshold}), falling back to random sampling")
        return unlabelled_candidates.sample(n=min(sample_size, len(unlabelled_candidates)), random_state=42)

    selected_indices = [idx for idx, score in top_candidates]
    selected_candidates = unlabelled_candidates.iloc[selected_indices].copy()
    similarity_scores = [score for idx, score in top_candidates]
    selected_candidates['similarity_score'] = similarity_scores

    print(f"  Selected {len(selected_candidates)} similar candidates (avg similarity: {np.mean(similarity_scores):.3f})")

    return selected_candidates

In [ ]:
def get_subreddit_proportions():
    prop_df = pd.concat((df, df_train))
    proportions = {}
    for rule in prop_df["rule"].unique():
        rule_data = prop_df[prop_df["rule"] == rule]
        subreddit_counts = rule_data["subreddit"].value_counts()
        subreddit_props = subreddit_counts / subreddit_counts.sum()
        proportions[rule] = subreddit_props.to_dict()
    return proportions, len(prop_df)

def sample_unlabelled_data_semantic(multiplier=5, similarity_threshold=0.3,
                                    embedding_model_name=f"{noizersam_all_minilm_l12_v2_pytorch_l12_v2_1_path}"):
    print(f"Loading embedding model: {embedding_model_name}")
    embedding_model = SentenceTransformer(embedding_model_name, device="cuda")

    print("Extracting positive examples from augmented_df...")
    positives_by_rule = get_positives_from_augmented_df(augmented_df)

    proportions, test_size = get_subreddit_proportions()
    total_sample_size = test_size * multiplier

    rule_counts = augmented_df["rule"].value_counts()
    rule_proportions = rule_counts / rule_counts.sum()

    excluded_values = augmented_df['body'].values
    global unlabelled_df
    unlabelled_df = unlabelled_df.query('body not in @excluded_values')
    unlabelled_df.drop(unlabelled_df[(unlabelled_df['body'].str.len() > 2000)].index, inplace=True)
    unlabelled_df['body_lower'] = unlabelled_df.body.str.lower()
    unlabelled_df.drop_duplicates(subset=['body_lower'], keep='first', inplace=True, ignore_index=True)
    unlabelled_df.drop(columns=['body_lower'], inplace=True)

    sampled_data = []

    print("Performing semantic similarity-based sampling...")
    for rule, rule_prop in tqdm(rule_proportions.items(), desc="Processing rules"):
        rule_sample_size = int(total_sample_size * rule_prop)
        subreddit_props = proportions[rule]
        positive_examples = positives_by_rule.get(rule, [])

        if not positive_examples:
            print(f"  No positive examples for rule: {rule[:50]}...")
            continue

        rule_samples = []
        for subreddit, sub_prop in subreddit_props.items():
            subreddit_sample_size = int(rule_sample_size * sub_prop)

            if subreddit_sample_size == 0:
                continue

            subreddit_data = unlabelled_df[
                unlabelled_df["subreddit"].str.lower().str.strip() == subreddit.lower().strip()
            ]

            if len(subreddit_data) == 0:
                continue

            print(f"\n  Rule: {rule[:30]}... | Subreddit: {subreddit} | Target: {subreddit_sample_size}")

            similar_samples = find_similar_unlabelled_examples(
                positive_examples=positive_examples,
                unlabelled_candidates=subreddit_data,
                embedding_model=embedding_model,
                sample_size=subreddit_sample_size,
                similarity_threshold=similarity_threshold
            )

            if len(similar_samples) > 0:
                similar_samples = similar_samples.copy()
                similar_samples["rule"] = rule
                rule_samples.append(similar_samples)

        if rule_samples:
            sampled_data.append(pd.concat(rule_samples, axis=0))

    if sampled_data:
        final_sample = pd.concat(sampled_data, axis=0).reset_index(drop=True)
        final_sample = final_sample.sample(frac=1, random_state=42).reset_index(drop=True)

        print(f"\n=== Sampling Results ===")
        print(f"Total samples: {len(final_sample)}")
        if 'similarity_score' in final_sample.columns:
            print(f"Average similarity: {final_sample['similarity_score'].mean():.3f}")
            print(f"Similarity range: {final_sample['similarity_score'].min():.3f} - {final_sample['similarity_score'].max():.3f}")

        return final_sample
    else:
        print("No samples found!")
        return pd.DataFrame()

In [ ]:
sampled_unlabelled = sample_unlabelled_data_semantic(multiplier=50, similarity_threshold=0.3)
print(f"\nSampled {len(sampled_unlabelled)} unlabelled rows")
print(sampled_unlabelled.head())

Loading embedding model: /kaggle/input/all-minilm-l12-v2/pytorch/l12-v2/1
Extracting positive examples from augmented_df...
Rule 'No Advertising: Spam, referral links, unsolicited ...': 385 positive examples
Rule 'No legal advice: Do not offer or request legal adv...': 591 positive examples
Performing semantic similarity-based sampling...


Processing rules: 0it [00:00, ?it/s]


  Rule: No legal advice: Do not offer ... | Subreddit: legaladvice | Target: 11362
  Encoding 591 positives and 13080 candidates...
  Selected 11102 similar candidates (avg similarity: 0.350)

  Rule: No legal advice: Do not offer ... | Subreddit: relationships | Target: 5681
  Encoding 591 positives and 50000 candidates...
  Selected 5681 similar candidates (avg similarity: 0.384)

  Rule: No legal advice: Do not offer ... | Subreddit: personalfinance | Target: 5518
  Encoding 591 positives and 22108 candidates...
  Selected 5518 similar candidates (avg similarity: 0.372)

  Rule: No legal advice: Do not offer ... | Subreddit: TwoXChromosomes | Target: 4382
  Encoding 591 positives and 49059 candidates...
  Selected 4382 similar candidates (avg similarity: 0.391)

  Rule: No legal advice: Do not offer ... | Subreddit: The_Donald | Target: 3949
  Encoding 591 positives and 50000 candidates...
  Selected 3949 similar candidates (avg similarity: 0.384)

  Rule: No legal advice: Do not o

Processing rules: 1it [28:23, 1703.76s/it]

  Selected 54 similar candidates (avg similarity: 0.403)

  Rule: No Advertising: Spam, referral... | Subreddit: soccerstreams | Target: 6334
  Encoding 385 positives and 6635 candidates...
  Selected 5838 similar candidates (avg similarity: 0.372)

  Rule: No Advertising: Spam, referral... | Subreddit: AskReddit | Target: 5508
  Encoding 385 positives and 50000 candidates...
  Selected 5508 similar candidates (avg similarity: 0.369)

  Rule: No Advertising: Spam, referral... | Subreddit: movies | Target: 2478
  Encoding 385 positives and 8762 candidates...
  Selected 2478 similar candidates (avg similarity: 0.354)

  Rule: No Advertising: Spam, referral... | Subreddit: videos | Target: 2111
  Encoding 385 positives and 11992 candidates...
  Selected 2111 similar candidates (avg similarity: 0.361)

  Rule: No Advertising: Spam, referral... | Subreddit: pics | Target: 1331
  Encoding 385 positives and 14816 candidates...
  Selected 1331 similar candidates (avg similarity: 0.375)

  Rule

Processing rules: 2it [56:26, 1693.15s/it]

  Selected 45 similar candidates (avg similarity: 0.457)

=== Sampling Results ===
Total samples: 101105
Average similarity: 0.392
Similarity range: 0.300 - 0.996

Sampled 101105 unlabelled rows
                                                body      subreddit  \
0  Its probably illegal, but Id do it. She may bl...    legaladvice   
1  Honestly? The jobs are going to decide this. \...  relationships   
2                                        "Uplifting"  UpliftingNews   
3   And you get this medical info from....your bum?      The_Donald   
4                              got a code message me    hearthstone   

   similarity_score                                               rule  
0          0.377491  No legal advice: Do not offer or request legal...  
1          0.402387  No legal advice: Do not offer or request legal...  
2          0.507457  No legal advice: Do not offer or request legal...  
3          0.466199  No legal advice: Do not offer or request legal...  
4          0.

In [ ]:
sampled_unlabelled.to_csv('sampled_unlabelled_100k.csv', index=False)
print(f"Saved sampled data to sampled_unlabelled_100k.csv")
print(f"Columns: {sampled_unlabelled.columns.tolist()}")

Saved sampled data to sampled_unlabelled_100k.csv
Columns: ['body', 'subreddit', 'similarity_score', 'rule']


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("\n" + "="*70)
print("CLEANING UP BASE MODEL - FULL GPU RESET")
print("="*70)

# Delete all base model objects
try:
    del model
except: pass
try:
    del tokenizer
except: pass
try:
    del train_loader, val_loader, test_loader
except: pass
try:
    del train_ds, val_ds, test_ds
except: pass
try:
    del optimizer, scheduler
except: pass

# Clear all CUDA state
torch.cuda.empty_cache()
torch.cuda.synchronize()
import gc
gc.collect()

# Reset CUDA device
torch.cuda.reset_peak_memory_stats()

print("GPU memory cleanup complete")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
sampled_unlabelled= pd.read_csv('/content/drive/MyDrive/sampled_unlabelled_100k.csv')



CLEANING UP BASE MODEL - FULL GPU RESET
GPU memory cleanup complete
GPU memory allocated: 0.00 GB


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def get_examples_by_rule(train_df):
    examples_by_rule = {}

    for rule in train_df['rule'].unique():
        rule_data = train_df[train_df['rule'] == rule]

        positives = rule_data[rule_data['rule_violation'] == 1]['body'].tolist()
        negatives = rule_data[rule_data['rule_violation'] == 0]['body'].tolist()

        pos_examples = rule_data[rule_data['rule_violation'] == 1][
            ['positive_example_1', 'positive_example_2']
        ].values.flatten().tolist()
        pos_examples = [p for p in pos_examples if pd.notna(p) and str(p).strip() != '']
        positives.extend(pos_examples)
        positives = list(set(positives))

        neg_examples = rule_data[rule_data['rule_violation'] == 0][
            ['negative_example_1', 'negative_example_2']
        ].values.flatten().tolist()
        neg_examples = [n for n in neg_examples if pd.notna(n) and str(n).strip() != '']
        negatives.extend(neg_examples)
        negatives = list(set(negatives))

        examples_by_rule[rule] = {
            'positives': positives,
            'negatives': negatives,
            'subreddit': rule_data['subreddit'].iloc[0] if len(rule_data) > 0 else ''
        }

        print(f"Rule '{rule[:50]}...': {len(positives)} positives, {len(negatives)} negatives")

    return examples_by_rule

examples_by_rule = get_examples_by_rule(df_train)
print(f"\nTotal rules: {len(examples_by_rule)}")

Rule 'No Advertising: Spam, referral links, unsolicited ...': 387 positives, 485 negatives
Rule 'No legal advice: Do not offer or request legal adv...': 591 positives, 423 negatives

Total rules: 2


In [ ]:
SYS_PROMPT = """
You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
"""

In [ ]:
llm = vllm.LLM(
    MODEL_NAME,
    # quantization='awq',
    quantization='gptq',
    tensor_parallel_size=torch.cuda.device_count(),
    gpu_memory_utilization=0.98,
    trust_remote_code=True,
    dtype="half",
    enforce_eager=True,
    max_model_len=4096,
    disable_log_stats=True,
    enable_prefix_caching=True,
    enable_lora=True,
    max_lora_rank=32
)
llm

tokenizer = llm.get_tokenizer()
tokenizer

mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])
mclp

INFO 10-05 05:38:28 [__init__.py:235] Automatically detected platform cuda.


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


INFO 10-05 05:38:51 [config.py:1604] Using max model len 4096
INFO 10-05 05:38:53 [gptq_marlin.py:174] Detected that the model can run with gptq_marlin, however you specified quantization=gptq explicitly, so forcing gptq. Use quantization=gptq_marlin for faster inference
WARNING 10-05 05:38:53 [config.py:1084] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 10-05 05:38:56 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 10-05 05:38:56 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4', speculative_config=None, tokenizer='Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

INFO 10-05 05:40:14 [weight_utils.py:312] Time spent downloading weights for Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4: 69.182189 seconds


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 10-05 05:40:18 [default_loader.py:262] Loading weights took 3.21 seconds
INFO 10-05 05:40:18 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 10-05 05:40:19 [model_runner.py:1115] Model loading took 9.6361 GiB and 74.270855 seconds
INFO 10-05 05:40:29 [worker.py:295] Memory profiling takes 10.00 seconds
INFO 10-05 05:40:29 [worker.py:295] the current vLLM instance can use total_gpu_memory (22.16GiB) x gpu_memory_utilization (0.98) = 21.72GiB
INFO 10-05 05:40:29 [worker.py:295] model weights take 9.64GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 1.43GiB; the rest of the memory reserved for KV Cache is 10.61GiB.
INFO 10-05 05:40:30 [executor_base.py:113] # cuda blocks: 3620, # CPU blocks: 1365
INFO 10-05 05:40:30 [executor_base.py:118] Maximum concurrency for 4096 tokens per request: 14.14x
INFO 10-05 05:40:33 [llm_engine.py:424] init engine (profile, create kv cache, warmup model) took 13.92 seconds


In [ ]:
import random

def create_prompts_for_unlabelled(df_unlabelled, examples_by_rule, tokenizer, sys_prompt):
    prompts = []

    for i, row in df_unlabelled.iterrows():
        rule = row['rule']
        subreddit = row['subreddit']
        body = row['body']

        if rule not in examples_by_rule:
            print(f"Warning: Rule '{rule[:30]}...' not found in examples_by_rule")
            continue

        examples = examples_by_rule[rule]

        if len(examples['positives']) < 2 or len(examples['negatives']) < 2:
            pos1 = examples['positives'][0] if len(examples['positives']) > 0 else "N/A"
            pos2 = examples['positives'][1] if len(examples['positives']) > 1 else examples['positives'][0] if len(examples['positives']) > 0 else "N/A"
            neg1 = examples['negatives'][0] if len(examples['negatives']) > 0 else "N/A"
            neg2 = examples['negatives'][1] if len(examples['negatives']) > 1 else examples['negatives'][0] if len(examples['negatives']) > 0 else "N/A"
        else:
            pos_samples = random.sample(examples['positives'], 2)
            neg_samples = random.sample(examples['negatives'], 2)
            pos1, pos2 = pos_samples
            neg1, neg2 = neg_samples

        text = f"""
r/{subreddit}
Rule: {rule}

1) {pos1}
Violation: Yes

2) {pos2}
Violation: Yes

3) {neg1}
Violation: No

4) {neg2}
Violation: No

5) {body}
"""

        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": text}
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + "Answer:"
        prompts.append(prompt)

    return prompts

random.seed(42)
unlabelled_prompts = create_prompts_for_unlabelled(
    sampled_unlabelled,
    examples_by_rule,
    tokenizer,
    SYS_PROMPT
)

print(f"Created {len(unlabelled_prompts)} prompts for unlabelled data")
print(f"\nFirst prompt sample:\n{unlabelled_prompts[0][:500]}...")

Created 101105 prompts for unlabelled data

First prompt sample:
<|im_start|>system

You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
<|im_end|>
<|im_start|>user

r/legaladvice
Rule: No legal advice: Do not offer or request legal advice.

1) fair enough, current civil and potentially criminal. The potential for jail is definitely there though I mean I dont see how running an entire company built around criminal acts can not lead to criminal charges. Unless your a rich billionaire of course maybe.
...


# 3.5. Inference on Unlabelled Data

In [ ]:
unlabelled_outputs = llm.generate(
    unlabelled_prompts,
    vllm.SamplingParams(
        skip_special_tokens=True,
        max_tokens=1,
        logits_processors=[mclp],
        logprobs=2,
    ),
    use_tqdm=True,
    lora_request=LoRARequest("default", 1, LORA_PATH)
)
print(f"Generated {len(unlabelled_outputs)} predictions")

Adding requests:   0%|          | 0/101105 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/101105 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks…

Generated 101105 predictions


In [ ]:
unlabelled_logprobs = [
    {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
    for out in unlabelled_outputs
]

unlabelled_logit_matrix = pd.DataFrame(unlabelled_logprobs)[['Yes','No']]
print(f"Logit matrix shape: {unlabelled_logit_matrix.shape}")
unlabelled_logit_matrix.head()

Logit matrix shape: (101105, 2)


,Yes,No
0,-0.151214,-1.963714
1,-1.256755,-0.334880
2,-3.910852,-0.020227
3,-3.499430,-0.030680
4,-2.665828,-0.072078


In [ ]:
unlabelled_probs = unlabelled_logit_matrix.apply(lambda x: softmax(x.values), axis=1, result_type="expand")
unlabelled_probs.columns = ['Yes', 'No']

sampled_unlabelled_with_preds = sampled_unlabelled.copy()
sampled_unlabelled_with_preds['pred_yes'] = unlabelled_probs['Yes']
sampled_unlabelled_with_preds['pred_no'] = unlabelled_probs['No']
sampled_unlabelled_with_preds['rule_violation'] = unlabelled_probs['Yes']

print(f"Predictions shape: {sampled_unlabelled_with_preds.shape}")
print(f"\nPrediction distribution:")
print(f"Mean: {sampled_unlabelled_with_preds['rule_violation'].mean():.4f}")
print(f"Median: {sampled_unlabelled_with_preds['rule_violation'].median():.4f}")
print(f"Min: {sampled_unlabelled_with_preds['rule_violation'].min():.4f}")
print(f"Max: {sampled_unlabelled_with_preds['rule_violation'].max():.4f}")

sampled_unlabelled_with_preds.head()

Predictions shape: (101105, 7)

Prediction distribution:
Mean: 0.2365
Median: 0.0575
Min: 0.0044
Max: 0.9956


,body,subreddit,similarity_score,rule,pred_yes,pred_no,rule_violation
0,"Its probably illegal, but Id do it. She may bl...",legaladvice,0.377491,No legal advice: Do not offer or request legal...,0.859664,0.140336,0.859664
1,Honestly? The jobs are going to decide this. \...,relationships,0.402387,No legal advice: Do not offer or request legal...,0.284576,0.715424,0.284576
2,"""Uplifting""",UpliftingNews,0.507457,No legal advice: Do not offer or request legal...,0.020023,0.979977,0.020023
3,And you get this medical info from....your bum?,The_Donald,0.466199,No legal advice: Do not offer or request legal...,0.030215,0.969785,0.030215
4,got a code message me,hearthstone,0.417036,"No Advertising: Spam, referral links, unsolici...",0.069542,0.930458,0.069542


In [ ]:
sampled_unlabelled_with_preds.to_csv('sampled_unlabelled_100k_with_predictions.csv', index=False)
print(f"Saved predictions to sampled_unlabelled_100k_with_predictions.csv")
print(f"Columns: {sampled_unlabelled_with_preds.columns.tolist()}")

Saved predictions to sampled_unlabelled_100k_with_predictions.csv
Columns: ['body', 'subreddit', 'similarity_score', 'rule', 'pred_yes', 'pred_no', 'rule_violation']


### RRR

In [ ]:
prompts = []
for i, row in df.iterrows():
    text = f"""
r/{row.subreddit}
Rule: {row.rule}

1) {row.positive_example_1}
Violation: Yes

2) {row.positive_example_2}
Violation: Yes

3) {row.negative_example_1}
Violation: No

4) {row.negative_example_2}
Violation: No

5) {row.body}
"""

    messages = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    ) + "Answer:"
    prompts.append(prompt)

df["prompt"] = prompts
df.shape

In [ ]:
print(df.iloc[0]['prompt'])

# 4. Inference

In [ ]:
outputs = llm.generate(
    prompts,
    vllm.SamplingParams(
        skip_special_tokens=True,
        max_tokens=1,
        logits_processors=[mclp],
        logprobs=2,
    ),
    use_tqdm=True,
    lora_request=LoRARequest("default", 1, LORA_PATH)
)
len(outputs)

In [ ]:
outputs[0]

In [ ]:
logprobs = [
    {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
    for out in outputs
]
len(logprobs)

In [ ]:
logprobs[0]

# 5. Submit

In [ ]:
logit_matrix = pd.DataFrame(logprobs)[['Yes','No']]
print(logit_matrix.shape)
logit_matrix.head()

In [ ]:
df = pd.concat([df, logit_matrix], axis=1)
print(df.shape)
df.head()

In [ ]:
df[['Yes',"No"]].head()

In [ ]:
df[['Yes',"No"]] = df[['Yes',"No"]].apply(lambda x: softmax(x.values), axis=1, result_type="expand")
df[['Yes',"No"]].head()

In [ ]:
df["pred"] = df["Yes"]
df['rule_violation'] = df["pred"]
df[['row_id', 'rule_violation']].to_csv("submission.csv",index=False)

In [ ]:
print(df[['row_id', 'rule_violation']].shape)
df[['row_id', 'rule_violation']].head()

In [ ]:
!head submission.csv

# Task
Modify the code to sample unlabelled data from a user-provided list of subreddits, using the maximum available data. Then, use two provided example pairs to create prompts for the LLM, incorporating a rule derived from the sampled unlabelled data. Finally, run inference on the unlabelled data and generate a submission file.

## Modify data sampling

### Subtask:
Update the `sample_unlabelled_data_semantic` function or create a new function to sample data from a user-defined list of subreddits. Ensure it samples the maximum available data from these subreddits.


**Reasoning**:
Define a list of target subreddits, filter the unlabelled data, and modify the sampling function to sample a fixed large number of rows from the filtered data for each rule, using the existing similarity search logic.



In [ ]:
TARGET_SUBREDDITS = [
    'legaladvice',
    'relationships',
    'personalfinance',
    'TwoXChromosomes',
    'The_Donald',
    'news',
    'politics',
    'AskReddit',
    'science',
    'worldnews',
    'hillaryclinton',
    'sex',
    'depression',
    'explainlikeimfive',
    'canada',
    'EnoughTrumpSpam',
    'whatisthisthing',
    'CFB',
    'LifeProTips',
    'creepyPMs',
    'nottheonion',
    'SandersForPresident',
    'NeutralPolitics',
    'SuicideWatch',
    'pokemongo',
    'UpliftingNews',
    'europe',
    'nosleep',
    'pcmasterrace',
    'DIY',
    'videos',
    'funny',
    'GlobalOffensiveTrade',
    'BlackPeopleTwitter',
    'Overwatch',
    'socialism',
    'tifu',
    'CanadaPolitics',
    'Games',
    'syriancivilwar',
    'movies',
    'ShitRedditSays',
    'PoliticalDiscussion',
    'gonewild',
    'Incels',
    'conspiracy',
    'AskTrumpSupporters',
    'television',
    'pics',
    'spacex',
    'MMA',
    'GlobalOffensive',
    'Christianity',
    'anime',
    'philosophy',
    'Android',
    'history',
    'books',
    'SubredditDrama',
    'askscience',
    'wow',
    'soccerstreams',
    'gameofthrones',
    'AskWomen',
    'hearthstone',
    'pokemon',
    'churning',
    'gifs',
    'aww',
    'Showerthoughts',
    'leagueoflegends',
    'gaming',
    'NSFW_GIF',
    'jailbreak',
    'dataisbeautiful',
    'OldSchoolCool',
    'Futurology',
    'hiphopheads',
    'nba',
    'GetMotivated',
    'space',
    'food',
    'india',
    'technology',
    'DestinyTheGame',
    'photoshopbattles'
]

def sample_unlabelled_data_semantic_filtered(multiplier=50, similarity_threshold=0.3,
                                              embedding_model_name=f"{noizersam_all_minilm_l12_v2_pytorch_l12_v2_1_path}",
                                              target_subreddits=None, max_samples_per_rule=50000):
    print(f"Loading embedding model: {embedding_model_name}")
    embedding_model = SentenceTransformer(embedding_model_name, device="cuda")

    print("Extracting positive examples from augmented_df...")
    positives_by_rule = get_positives_from_augmented_df(augmented_df)

    excluded_values = augmented_df['body'].values
    global unlabelled_df
    unlabelled_df = unlabelled_df.query('body not in @excluded_values')
    unlabelled_df.drop(unlabelled_df[(unlabelled_df['body'].str.len() > 2000)].index, inplace=True)
    unlabelled_df['body_lower'] = unlabelled_df.body.str.lower()
    unlabelled_df.drop_duplicates(subset=['body_lower'], keep='first', inplace=True, ignore_index=True)
    unlabelled_df.drop(columns=['body_lower'], inplace=True)

    if target_subreddits:
        print(f"Filtering unlabelled data for target subreddits: {len(target_subreddits)}")
        filtered_unlabelled_df = unlabelled_df[unlabelled_df['subreddit'].isin(target_subreddits)].copy()
    else:
        filtered_unlabelled_df = unlabelled_df.copy()

    print(f"Filtered unlabelled data shape: {filtered_unlabelled_df.shape}")

    sampled_data = []

    print("Performing semantic similarity-based sampling on filtered data...")
    for rule, positives in tqdm(positives_by_rule.items(), desc="Processing rules"):
        if not positives:
            print(f"  No positive examples for rule: {rule[:50]}...")
            continue

        # Sample a fixed large number of candidates from the filtered data for this rule
        rule_candidates = filtered_unlabelled_df.sample(
            n=min(max_samples_per_rule, len(filtered_unlabelled_df)),
            replace=False,
            random_state=42
        )

        if len(rule_candidates) == 0:
            continue

        print(f"\n  Rule: {rule[:30]}... | Candidates: {len(rule_candidates)}")

        similar_samples = find_similar_unlabelled_examples(
            positive_examples=positives,
            unlabelled_candidates=rule_candidates,
            embedding_model=embedding_model,
            sample_size=min(max_samples_per_rule // len(positives_by_rule), len(rule_candidates)), # Adjust sample size based on rule count
            similarity_threshold=similarity_threshold
        )

        if len(similar_samples) > 0:
            similar_samples = similar_samples.copy()
            similar_samples["rule"] = rule
            sampled_data.append(similar_samples)

    if sampled_data:
        final_sample = pd.concat(sampled_data, axis=0).reset_index(drop=True)
        final_sample = final_sample.sample(frac=1, random_state=42).reset_index(drop=True)

        print(f"\n=== Sampling Results ===")
        print(f"Total samples: {len(final_sample)}")
        if 'similarity_score' in final_sample.columns:
            print(f"Average similarity: {final_sample['similarity_score'].mean():.3f}")
            print(f"Similarity range: {final_sample['similarity_score'].min():.3f} - {final_sample['similarity_score'].max():.3f}")

        return final_sample
    else:
        print("No samples found!")
        return pd.DataFrame()
